# Phase 2: Text Preprocessing

This notebook prepares the cleaned mental health text dataset for feature engineering and machine learning. The workflow includes inspecting text quality, repairing character encoding artifacts, normalizing text, tokenizing, removing selected stop words, applying lemmatization, validating the results, and saving a preprocessed dataset.

In [1]:
from html import unescape
from pathlib import Path
import re

import ftfy
import pandas as pd

## Load processed dataset

In [2]:
PROCESSED_DATA_DIR = Path.cwd().parent / "data" / "processed"

input_file = PROCESSED_DATA_DIR / "mental_health_text_dataset.csv"

mental_health_df = pd.read_csv(input_file)

print("Dataset shape:", mental_health_df.shape)
mental_health_df.head()

Dataset shape: (17699, 2)


,Label,TEXT
0,0.0,TIL the movie Starship Troopers was never adap...
1,0.0,What do you call a fat baby?
2,0.0,Two morons are sitting on a fence. The big one...
3,0.0,I covered all my weapons in glue.
4,0.0,Joke I made up: Caveman and a bear walk into a...


## Explore the Text

### View random records

In [3]:
mental_health_df.sample(10, random_state=42)

,Label,TEXT
980,0.0,What fruit must get married locally?
5157,0.0,_ara Good to hear that Allah aapko sehat de (Y)
7359,1.0,I cannot survive the dissolution with my loved...
14814,2.0,"I'm a teen. A teen girl.\nI get mood swings, w..."
2805,0.0,"""Here, next to mine"" wasn't the answer i was e..."
7487,1.0,. I want to tell you my story ...
16754,2.0,I'm struggling.\nI'm actually fucking struggli...
11584,2.0,goodbye message
4203,0.0,I can`t really say much because none of it is...
11000,2.0,I just don't feel like living really.


### Review text length

In [4]:
mental_health_df["character_count"] = mental_health_df["TEXT"].str.len()
mental_health_df["word_count"] = mental_health_df["TEXT"].str.split().str.len()
mental_health_df[["character_count", "word_count"]].describe()

,character_count,word_count
count,17699.000000,17699.000000
mean,572.167976,108.001017
std,1075.965746,191.214970
min,1.000000,1.000000
25%,44.000000,9.000000
50%,135.000000,27.000000
75%,719.000000,140.000000
max,25558.000000,4834.000000


### View the shortest records

In [5]:
mental_health_df.nsmallest(20, "word_count")[["Label", "TEXT", "word_count"]]

,Label,TEXT,word_count
58,0.0,Threesome?,1
251,0.0,Ants,1
255,0.0,Botany,1
661,0.0,V,1
714,0.0,Soldiers,1
797,0.0,#NAME?,1
932,0.0,Pythagoras,1
1148,0.0,Pony,1
1267,0.0,Farmer,1
1327,0.0,Infrequently,1


## Inspect text quality

### Check Missing Values


In [6]:
mental_health_df.isnull().sum()

Label              0
TEXT               0
character_count    0
word_count         0
dtype: int64

### Count Duplicate Text

In [7]:
duplicate_text_count = mental_health_df.duplicated(subset=["TEXT"]).sum()

print("Duplicated TEXT values:", duplicate_text_count)

Duplicated TEXT values: 1834


### Review Conflicting Labels

In [8]:
text_label_counts = mental_health_df.groupby("TEXT")["Label"].nunique()

conflicting_text_count = (text_label_counts > 1).sum()

print(
    "TEXT values assigned to more than one label:",
    conflicting_text_count,
)

TEXT values assigned to more than one label: 4


### Review Duplicate Examples

In [9]:
duplicate_examples = mental_health_df[
    mental_health_df.duplicated(subset=["TEXT"], keep=False)
].sort_values("TEXT")

duplicate_examples.head(20)

,Label,TEXT,character_count,word_count
8681,1.0,.. the story is probably banal to the extreme...,1446,284
8431,1.0,.. the story is probably banal to the extreme...,1446,284
8385,1.0,"... Now there will be a lot of things, but I ...",132,27
8635,1.0,"... Now there will be a lot of things, but I ...",132,27
16686,2.0,This posting has been approved by the moderat...,2019,341
13317,2.0,This posting has been approved by the moderat...,2019,341
8347,1.0,dear readers.,14,2
8597,1.0,dear readers.,14,2
9200,1.0,everybody.,11,1
9336,1.0,everybody.,11,1


### Display Records with Conflicting Labels

In [10]:
# Find TEXT values assigned to more than one label
conflicting_posts = text_label_counts[text_label_counts > 1].index

# Display all records with conflicting labels
mental_health_df[mental_health_df["TEXT"].isin(conflicting_posts)].sort_values(
    ["TEXT", "Label"]
)[["Label", "TEXT", "character_count", "word_count"]]

,Label,TEXT,character_count,word_count
3330,0.0,.,1,1
7681,1.0,.,1,1
15265,2.0,.,1,1
16512,2.0,.,1,1
1873,0.0,Alone.,6,1
11941,2.0,Alone.,6,1
7810,1.0,I hate myself.,14,3
11815,2.0,I hate myself.,14,3
7190,1.0,I'm tired,9,2
11742,2.0,I'm tired,9,2


### Remove Conflicting Label Records

In [11]:
mental_health_df = mental_health_df[
    ~mental_health_df["TEXT"].isin(conflicting_posts)
].copy()

### Remove Duplicate Records

In [12]:
mental_health_df = mental_health_df.drop_duplicates(subset=["TEXT"]).copy()

print("Shape after removing duplicate text:")
print(mental_health_df.shape)

Shape after removing duplicate text:
(15861, 4)


### Duplicate Assessment Summary

An inspection identified 1,834 duplicate text records. Four text values were assigned to multiple target labels, including ".", "Alone.", "I hate myself.", and "I'm tired". Because identical text associated with different labels introduces ambiguity into supervised machine learning, these conflicting records were removed prior to preprocessing.

After removing conflicting records and duplicate social media posts with matching labels, the dataset contained 15,861 unique observations for preprocessing.

## Inspect and Repair Encoding Artifacts

### Identify Potential Encoding Artifacts

In [13]:
# Gives count and sample of the affected records
encoding_artifact_mask = mental_health_df["TEXT"].str.contains(
    r"Ã|Â|â|ð|�|\?\?\?",
    regex=True,
    na=False,
)

print(
    "Records with possible encoding artifacts:",
    encoding_artifact_mask.sum(),
)

mental_health_df.loc[
    encoding_artifact_mask,
    ["Label", "TEXT"],
].head(20)

Records with possible encoding artifacts: 1800


,Label,TEXT
15,0.0,"I???m in so much debt, I can???t afford to pay..."
19,0.0,A Woman Shoots Her Husband For Stepping On The...
31,0.0,"I said to my wife, ???I need to call the docto..."
76,0.0,Why did the roofer visit the doctor???s office...
84,0.0,I???m addicted to seaweed.
124,0.0,What???s your stance on 9/11?
146,0.0,What???s the difference between roast beef and...
170,0.0,If at first you don???t succeed...
197,0.0,???Squeeze 18 lemons and drink the juice all a...
231,0.0,"???Darling, can I go out in this dress???"


### Preserve the Original Text

In [14]:
mental_health_df["TEXT_ORIGINAL"] = mental_health_df["TEXT"]

### Repair Character Encoding

In [15]:
mental_health_df["TEXT_REPAIRED"] = (
    mental_health_df["TEXT_ORIGINAL"].astype(str).apply(ftfy.fix_text)
)

### Compare Original and Repaired Text

In [16]:
mental_health_df.loc[
    encoding_artifact_mask,
    ["TEXT_ORIGINAL", "TEXT_REPAIRED"],
].head(20)

,TEXT_ORIGINAL,TEXT_REPAIRED
15,"I???m in so much debt, I can???t afford to pay...","I???m in so much debt, I can???t afford to pay..."
19,A Woman Shoots Her Husband For Stepping On The...,A Woman Shoots Her Husband For Stepping On The...
31,"I said to my wife, ???I need to call the docto...","I said to my wife, ???I need to call the docto..."
76,Why did the roofer visit the doctor???s office...,Why did the roofer visit the doctor???s office...
84,I???m addicted to seaweed.,I???m addicted to seaweed.
124,What???s your stance on 9/11?,What???s your stance on 9/11?
146,What???s the difference between roast beef and...,What???s the difference between roast beef and...
170,If at first you don???t succeed...,If at first you don???t succeed...
197,???Squeeze 18 lemons and drink the juice all a...,???Squeeze 18 lemons and drink the juice all a...
231,"???Darling, can I go out in this dress???","???Darling, can I go out in this dress???"


### Review Remaining Encoding Artifacts

In [17]:
remaining_artifact_mask = mental_health_df["TEXT_REPAIRED"].str.contains(
    r"Ã|Â|â|ð|�|\?\?\?",
    regex=True,
    na=False,
)

print(
    "Possible artifacts remaining after repair:",
    remaining_artifact_mask.sum(),
)

Possible artifacts remaining after repair: 488


### Inspect Sample Records

In [18]:
mental_health_df.loc[
    remaining_artifact_mask, ["TEXT_ORIGINAL", "TEXT_REPAIRED"]
].sample(20, random_state=42)

,TEXT_ORIGINAL,TEXT_REPAIRED
2105,"I don???t know, and I don???t care.","I don???t know, and I don???t care."
11599,What is the moast painless way to kill yoursel...,What is the moast painless way to kill yoursel...
16329,.I've been depressed probably from the ages of...,.I've been depressed probably from the ages of...
2383,I don???t like to be spoken to in that tone of...,I don???t like to be spoken to in that tone of...
15541,How should I feel if I reach out to somebody f...,How should I feel if I reach out to somebody f...
17550,I just want to end it and cause minimal harm t...,I just want to end it and cause minimal harm t...
7012,Help me please. My heart is so bad that someti...,Help me please. My heart is so bad that someti...
1875,???I???ll tell you tomorrow.??,???I???ll tell you tomorrow.??
2516,. ???It???s a huge event. Why aren???t you exc...,. ???It???s a huge event. Why aren???t you exc...
14536,Iï¿½Ûªm not really sure how much information i...,I�۪m not really sure how much information is g...


In [19]:
# Create the normalization patterns
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
EMAIL_PATTERN = re.compile(r"\S+@\S+\.\S+")
USER_PATTERN = re.compile(r"@\w+")
HTML_PATTERN = re.compile(r"<[^>]+>")

SOCIAL_MARKER_PATTERN = re.compile(
    r"(?:EDIT|UPDATE|TL\s*;?\s*DR|ETA|RT)\s*:?",
    re.IGNORECASE,
)

WHITESPACE_PATTERN = re.compile(r"\s+")

In [20]:
def normalize_text(text: str) -> str:
    """Repair common text issues and normalize basic formatting."""
    text = ftfy.fix_text(text)
    text = unescape(text)

    # Replace common damaged apostrophe and quotation patterns.
    text = text.replace("??", '"')
    text = text.replace("�۪", "'")

    # Remove web-specific content.
    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)
    text = USER_PATTERN.sub(" ", text)
    text = HTML_PATTERN.sub(" ", text)

    # Remove common social media markers.
    text = SOCIAL_MARKER_PATTERN.sub(" ", text)

    # Normalize whitespace.
    text = WHITESPACE_PATTERN.sub(" ", text).strip()

    return text

In [21]:
# Apply the function
mental_health_df["TEXT_NORMALIZED"] = (
    mental_health_df["TEXT_REPAIRED"].astype(str).apply(normalize_text)
)

In [22]:
# Compare the results
mental_health_df[["TEXT_ORIGINAL", "TEXT_NORMALIZED"]].sample(20, random_state=42)

,TEXT_ORIGINAL,TEXT_NORMALIZED
12033,I Hope I Fucking Die Today.,I Hope I Fucking Die Today.
3134,It's called the iAye.,It's called the iAye.
13051,I figured out that i cannot work. and money fr...,I figured out that i cannot work. and money fr...
1662,Picking up women is like chess...,Picking up women is like chess...
2481,Because they are more likely to be deadEDIT: W...,Because they are more likely to be dead Wow. N...
12911,After my recent failed suicide attempt I can't...,After my recent failed suicide attempt I can't...
903,I always buy groceries with cash.,I always buy groceries with cash.
13728,I have the razor ready to slice open my arms.,I have the razor ready to slice open my arms.
4282,What are you reading?,What are you reading?
5285,Now I`m off to bed - HAPPY MOTHER`S DAY ALL - ...,Now I`m off to bed - HAPPY MOTHER`S DAY ALL - ...


In [23]:
# Inspect the flagged rows
mental_health_df.loc[
    remaining_artifact_mask, ["TEXT_ORIGINAL", "TEXT_NORMALIZED"]
].head(20)

,TEXT_ORIGINAL,TEXT_NORMALIZED
15,"I???m in so much debt, I can???t afford to pay...","I???m in so much debt, I can???t afford to pay..."
19,A Woman Shoots Her Husband For Stepping On The...,A Woman Shoots Her Husband For Stepping On The...
31,"I said to my wife, ???I need to call the docto...","I said to my wife, ???I need to call the docto..."
76,Why did the roofer visit the doctor???s office...,Why did the roofer visit the doctor???s office...
84,I???m addicted to seaweed.,I???m addicted to seaweed.
124,What???s your stance on 9/11?,What???s your stance on 9/11?
146,What???s the difference between roast beef and...,What???s the difference between roast beef and...
170,If at first you don???t succeed...,If at first you don???t succeed...
197,???Squeeze 18 lemons and drink the juice all a...,???Squeeze 18 lemons and drink the juice all a...
231,"???Darling, can I go out in this dress???","???Darling, can I go out in this dress?"""


In [24]:
normalized_artifact_mask = mental_health_df["TEXT_NORMALIZED"].str.contains(
    r"Ã|Â|â|ð|�|\?\?\?",
    regex=True,
    na=False,
)

print(
    "Possible artifacts remaining after normalization:",
    normalized_artifact_mask.sum(),
)

Possible artifacts remaining after normalization: 371


### Normalization Observation

The source dataset contained several damaged punctuation and character patterns that could not be fully repaired automatically. Common apostrophe and quotation artifacts were normalized where possible. Records were retained when the surrounding text remained meaningful because removing them would discard useful information for classification.

## Tokenize the Text

### Load the spaCy Language Model

In [25]:
import spacy

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"],
)

print("spaCy language model loaded successfully.")

spaCy language model loaded successfully.


### Define Important Negation Terms

In [26]:
# These words shouldn't be removed because it can change the meaning of mental health text
NEGATION_WORDS = {
    "no",
    "not",
    "never",
    "neither",
    "nor",
}

### Create the Preprocessing Function

In [27]:
"""Function removes spaces and punctuation replaces numbers with number converts tokens to lowercase lemmas removes stop words preserves important negation words keeps alphabetic tokens only"""


def preprocess_doc(doc) -> str:
    processed_tokens = []

    for token in doc:
        if token.is_space or token.is_punct:
            continue

        if token.like_num:
            processed_tokens.append("number")
            continue

        lemma = token.lemma_.lower().strip()

        if token.is_stop and lemma not in NEGATION_WORDS:
            continue

        if lemma.isalpha():
            processed_tokens.append(lemma)

    return " ".join(processed_tokens)

### Apply Tokenization and Lemmatization

In [28]:
# Create a column named TEXT_PROCESSED
docs = nlp.pipe(
    mental_health_df["TEXT_NORMALIZED"],
    batch_size=500,
)

mental_health_df["TEXT_PROCESSED"] = [preprocess_doc(doc) for doc in docs]

### Compare Normalized and Processed Text

In [29]:
mental_health_df[["TEXT_NORMALIZED", "TEXT_PROCESSED", "Label"]].sample(
    10, random_state=42
)

,TEXT_NORMALIZED,TEXT_PROCESSED,Label
12033,I Hope I Fucking Die Today.,hope fucking die today,2.0
3134,It's called the iAye.,call iaye,0.0
13051,I figured out that i cannot work. and money fr...,figure not work money family get dry gf strugg...,2.0
1662,Picking up women is like chess...,pick woman like chess,0.0
2481,Because they are more likely to be dead Wow. N...,likely dead wow understand rip inbox,0.0
12911,After my recent failed suicide attempt I can't...,recent fail suicide attempt not anymore body w...,2.0
903,I always buy groceries with cash.,buy grocery cash,0.0
13728,I have the razor ready to slice open my arms.,razor ready slice open arm,2.0
4282,What are you reading?,read,0.0
5285,Now I`m off to bed - HAPPY MOTHER`S DAY ALL - ...,bed happy day great number,0.0


### Validate the Processed Text

In [30]:
# Check for missing processed text
mental_health_df["TEXT_PROCESSED"].isna().sum()

np.int64(0)

In [31]:
# Count empty processed documents
(mental_health_df["TEXT_PROCESSED"].str.strip() == "").sum()

np.int64(87)

# Phase 2: Text Preprocessing

This notebook prepares the cleaned mental health text dataset for feature engineering and machine learning. The workflow includes inspecting text quality, repairing character encoding artifacts, normalizing text, tokenizing, removing selected stop words, applying lemmatization, validating the results, and saving a preprocessed dataset.

In [32]:
from pathlib import Path
import re

import ftfy
import pandas as pd

## Load processed dataset

In [33]:
PROCESSED_DATA_DIR = Path.cwd().parent / "data" / "processed"

input_file = PROCESSED_DATA_DIR / "mental_health_text_dataset.csv"

mental_health_df = pd.read_csv(input_file)

print("Dataset shape:", mental_health_df.shape)
mental_health_df.head()

Dataset shape: (17699, 2)


,Label,TEXT
0,0.0,TIL the movie Starship Troopers was never adap...
1,0.0,What do you call a fat baby?
2,0.0,Two morons are sitting on a fence. The big one...
3,0.0,I covered all my weapons in glue.
4,0.0,Joke I made up: Caveman and a bear walk into a...


## Explore the Text

### View random records

In [34]:
mental_health_df.sample(10, random_state=42)

,Label,TEXT
980,0.0,What fruit must get married locally?
5157,0.0,_ara Good to hear that Allah aapko sehat de (Y)
7359,1.0,I cannot survive the dissolution with my loved...
14814,2.0,"I'm a teen. A teen girl.\nI get mood swings, w..."
2805,0.0,"""Here, next to mine"" wasn't the answer i was e..."
7487,1.0,. I want to tell you my story ...
16754,2.0,I'm struggling.\nI'm actually fucking struggli...
11584,2.0,goodbye message
4203,0.0,I can`t really say much because none of it is...
11000,2.0,I just don't feel like living really.


### Review text length

In [35]:
mental_health_df["character_count"] = mental_health_df["TEXT"].str.len()
mental_health_df["word_count"] = mental_health_df["TEXT"].str.split().str.len()
mental_health_df[["character_count", "word_count"]].describe()

,character_count,word_count
count,17699.000000,17699.000000
mean,572.167976,108.001017
std,1075.965746,191.214970
min,1.000000,1.000000
25%,44.000000,9.000000
50%,135.000000,27.000000
75%,719.000000,140.000000
max,25558.000000,4834.000000


### View the shortest records

In [36]:
mental_health_df.nsmallest(20, "word_count")[["Label", "TEXT", "word_count"]]

,Label,TEXT,word_count
58,0.0,Threesome?,1
251,0.0,Ants,1
255,0.0,Botany,1
661,0.0,V,1
714,0.0,Soldiers,1
797,0.0,#NAME?,1
932,0.0,Pythagoras,1
1148,0.0,Pony,1
1267,0.0,Farmer,1
1327,0.0,Infrequently,1


## Inspect text quality

### Check Missing Values


In [37]:
mental_health_df.isnull().sum()

Label              0
TEXT               0
character_count    0
word_count         0
dtype: int64

### Count Duplicate Text

In [38]:
duplicate_text_count = mental_health_df.duplicated(subset=["TEXT"]).sum()

print("Duplicated TEXT values:", duplicate_text_count)

Duplicated TEXT values: 1834


### Review Conflicting Labels

In [39]:
text_label_counts = mental_health_df.groupby("TEXT")["Label"].nunique()

conflicting_text_count = (text_label_counts > 1).sum()

print(
    "TEXT values assigned to more than one label:",
    conflicting_text_count,
)

TEXT values assigned to more than one label: 4


### Review Duplicate Examples

In [40]:
duplicate_examples = mental_health_df[
    mental_health_df.duplicated(subset=["TEXT"], keep=False)
].sort_values("TEXT")

duplicate_examples.head(20)

,Label,TEXT,character_count,word_count
8681,1.0,.. the story is probably banal to the extreme...,1446,284
8431,1.0,.. the story is probably banal to the extreme...,1446,284
8385,1.0,"... Now there will be a lot of things, but I ...",132,27
8635,1.0,"... Now there will be a lot of things, but I ...",132,27
16686,2.0,This posting has been approved by the moderat...,2019,341
13317,2.0,This posting has been approved by the moderat...,2019,341
8347,1.0,dear readers.,14,2
8597,1.0,dear readers.,14,2
9200,1.0,everybody.,11,1
9336,1.0,everybody.,11,1


### Display Records with Conflicting Labels

In [41]:
# Find TEXT values assigned to more than one label
conflicting_posts = text_label_counts[text_label_counts > 1].index

# Display all records with conflicting labels
mental_health_df[mental_health_df["TEXT"].isin(conflicting_posts)].sort_values(
    ["TEXT", "Label"]
)[["Label", "TEXT", "character_count", "word_count"]]

,Label,TEXT,character_count,word_count
3330,0.0,.,1,1
7681,1.0,.,1,1
15265,2.0,.,1,1
16512,2.0,.,1,1
1873,0.0,Alone.,6,1
11941,2.0,Alone.,6,1
7810,1.0,I hate myself.,14,3
11815,2.0,I hate myself.,14,3
7190,1.0,I'm tired,9,2
11742,2.0,I'm tired,9,2


### Remove Conflicting Label Records

In [42]:
mental_health_df = mental_health_df[
    ~mental_health_df["TEXT"].isin(conflicting_posts)
].copy()

### Remove Duplicate Records

In [43]:
mental_health_df = mental_health_df.drop_duplicates(subset=["TEXT"]).copy()

print("Shape after removing duplicate text:")
print(mental_health_df.shape)

Shape after removing duplicate text:
(15861, 4)


### Duplicate Assessment Summary

An inspection identified 1,834 duplicate text records. Four text values were assigned to multiple target labels, including ".", "Alone.", "I hate myself.", and "I'm tired". Because identical text associated with different labels introduces ambiguity into supervised machine learning, these conflicting records were removed prior to preprocessing.

After removing the conflicting records, duplicate social media posts with identical text and matching labels were removed, resulting in a final dataset of **15,861 unique observations** for text preprocessing.

## Inspect and Repair Encoding Artifacts

### Identify Potential Encoding Artifacts

In [44]:
# Gives count and sample of the affected records
encoding_artifact_mask = mental_health_df["TEXT"].str.contains(
    r"Ã|Â|â|ð|�|\?\?\?",
    regex=True,
    na=False,
)

print(
    "Records with possible encoding artifacts:",
    encoding_artifact_mask.sum(),
)

mental_health_df.loc[
    encoding_artifact_mask,
    ["Label", "TEXT"],
].head(20)

Records with possible encoding artifacts: 1800


,Label,TEXT
15,0.0,"I???m in so much debt, I can???t afford to pay..."
19,0.0,A Woman Shoots Her Husband For Stepping On The...
31,0.0,"I said to my wife, ???I need to call the docto..."
76,0.0,Why did the roofer visit the doctor???s office...
84,0.0,I???m addicted to seaweed.
124,0.0,What???s your stance on 9/11?
146,0.0,What???s the difference between roast beef and...
170,0.0,If at first you don???t succeed...
197,0.0,???Squeeze 18 lemons and drink the juice all a...
231,0.0,"???Darling, can I go out in this dress???"


### Preserve the Original Text

In [45]:
mental_health_df["TEXT_ORIGINAL"] = mental_health_df["TEXT"]

### Repair Character Encoding

In [46]:
mental_health_df["TEXT_REPAIRED"] = (
    mental_health_df["TEXT_ORIGINAL"].astype(str).apply(ftfy.fix_text)
)

### Compare Original and Repaired Text

In [47]:
mental_health_df.loc[
    encoding_artifact_mask,
    ["TEXT_ORIGINAL", "TEXT_REPAIRED"],
].head(20)

,TEXT_ORIGINAL,TEXT_REPAIRED
15,"I???m in so much debt, I can???t afford to pay...","I???m in so much debt, I can???t afford to pay..."
19,A Woman Shoots Her Husband For Stepping On The...,A Woman Shoots Her Husband For Stepping On The...
31,"I said to my wife, ???I need to call the docto...","I said to my wife, ???I need to call the docto..."
76,Why did the roofer visit the doctor???s office...,Why did the roofer visit the doctor???s office...
84,I???m addicted to seaweed.,I???m addicted to seaweed.
124,What???s your stance on 9/11?,What???s your stance on 9/11?
146,What???s the difference between roast beef and...,What???s the difference between roast beef and...
170,If at first you don???t succeed...,If at first you don???t succeed...
197,???Squeeze 18 lemons and drink the juice all a...,???Squeeze 18 lemons and drink the juice all a...
231,"???Darling, can I go out in this dress???","???Darling, can I go out in this dress???"


### Review Remaining Encoding Artifacts

In [48]:
remaining_artifact_mask = mental_health_df["TEXT_REPAIRED"].str.contains(
    r"Ã|Â|â|ð|�|\?\?\?",
    regex=True,
    na=False,
)

print(
    "Possible artifacts remaining after repair:",
    remaining_artifact_mask.sum(),
)

Possible artifacts remaining after repair: 488


### Inspect Sample Records

In [49]:
mental_health_df.loc[
    remaining_artifact_mask, ["TEXT_ORIGINAL", "TEXT_REPAIRED"]
].sample(20, random_state=42)

,TEXT_ORIGINAL,TEXT_REPAIRED
2105,"I don???t know, and I don???t care.","I don???t know, and I don???t care."
11599,What is the moast painless way to kill yoursel...,What is the moast painless way to kill yoursel...
16329,.I've been depressed probably from the ages of...,.I've been depressed probably from the ages of...
2383,I don???t like to be spoken to in that tone of...,I don???t like to be spoken to in that tone of...
15541,How should I feel if I reach out to somebody f...,How should I feel if I reach out to somebody f...
17550,I just want to end it and cause minimal harm t...,I just want to end it and cause minimal harm t...
7012,Help me please. My heart is so bad that someti...,Help me please. My heart is so bad that someti...
1875,???I???ll tell you tomorrow.??,???I???ll tell you tomorrow.??
2516,. ???It???s a huge event. Why aren???t you exc...,. ???It???s a huge event. Why aren???t you exc...
14536,Iï¿½Ûªm not really sure how much information i...,I�۪m not really sure how much information is g...


In [50]:
# Create the normalization patterns
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
EMAIL_PATTERN = re.compile(r"\S+@\S+\.\S+")
USER_PATTERN = re.compile(r"@\w+")
HTML_PATTERN = re.compile(r"<[^>]+>")

SOCIAL_MARKER_PATTERN = re.compile(
    r"(?:EDIT|UPDATE|TL\s*;?\s*DR|ETA|RT)\s*:?",
    re.IGNORECASE,
)

WHITESPACE_PATTERN = re.compile(r"\s+")

In [51]:
def normalize_text(text: str) -> str:
    """Repair common text issues and normalize basic formatting."""
    text = ftfy.fix_text(text)
    text = unescape(text)

    # Replace common damaged apostrophe and quotation patterns.
    text = text.replace("??", '"')
    text = text.replace("�۪", "'")

    # Remove web-specific content.
    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)
    text = USER_PATTERN.sub(" ", text)
    text = HTML_PATTERN.sub(" ", text)

    # Remove common social media markers.
    text = SOCIAL_MARKER_PATTERN.sub(" ", text)

    # Normalize whitespace.
    text = WHITESPACE_PATTERN.sub(" ", text).strip()

    return text

In [52]:
# Apply the function
mental_health_df["TEXT_NORMALIZED"] = (
    mental_health_df["TEXT_ORIGINAL"].astype(str).apply(normalize_text)
)

In [53]:
# Compare the results
mental_health_df[["TEXT_ORIGINAL", "TEXT_NORMALIZED"]].sample(20, random_state=42)

,TEXT_ORIGINAL,TEXT_NORMALIZED
12033,I Hope I Fucking Die Today.,I Hope I Fucking Die Today.
3134,It's called the iAye.,It's called the iAye.
13051,I figured out that i cannot work. and money fr...,I figured out that i cannot work. and money fr...
1662,Picking up women is like chess...,Picking up women is like chess...
2481,Because they are more likely to be deadEDIT: W...,Because they are more likely to be dead Wow. N...
12911,After my recent failed suicide attempt I can't...,After my recent failed suicide attempt I can't...
903,I always buy groceries with cash.,I always buy groceries with cash.
13728,I have the razor ready to slice open my arms.,I have the razor ready to slice open my arms.
4282,What are you reading?,What are you reading?
5285,Now I`m off to bed - HAPPY MOTHER`S DAY ALL - ...,Now I`m off to bed - HAPPY MOTHER`S DAY ALL - ...


In [54]:
# Inspect the flagged rows
mental_health_df.loc[
    remaining_artifact_mask, ["TEXT_ORIGINAL", "TEXT_NORMALIZED"]
].head(20)

,TEXT_ORIGINAL,TEXT_NORMALIZED
15,"I???m in so much debt, I can???t afford to pay...","I???m in so much debt, I can???t afford to pay..."
19,A Woman Shoots Her Husband For Stepping On The...,A Woman Shoots Her Husband For Stepping On The...
31,"I said to my wife, ???I need to call the docto...","I said to my wife, ???I need to call the docto..."
76,Why did the roofer visit the doctor???s office...,Why did the roofer visit the doctor???s office...
84,I???m addicted to seaweed.,I???m addicted to seaweed.
124,What???s your stance on 9/11?,What???s your stance on 9/11?
146,What???s the difference between roast beef and...,What???s the difference between roast beef and...
170,If at first you don???t succeed...,If at first you don???t succeed...
197,???Squeeze 18 lemons and drink the juice all a...,???Squeeze 18 lemons and drink the juice all a...
231,"???Darling, can I go out in this dress???","???Darling, can I go out in this dress?"""


In [55]:
normalized_artifact_mask = mental_health_df["TEXT_NORMALIZED"].str.contains(
    r"Ã|Â|â|ð|�|\?\?\?",
    regex=True,
    na=False,
)

print(
    "Possible artifacts remaining after normalization:",
    normalized_artifact_mask.sum(),
)

Possible artifacts remaining after normalization: 371


### Normalization Observation

The source dataset contained several damaged punctuation and character patterns that could not be fully repaired automatically. Common apostrophe and quotation artifacts were normalized where possible. Records were retained when the surrounding text remained meaningful because removing them would discard useful information for classification.

## Tokenize the Text

### Load the spaCy Language Model

In [56]:
import spacy

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"],
)

print("spaCy language model loaded successfully.")

spaCy language model loaded successfully.


### Define Important Negation Terms

In [57]:
# These words shouldn't be removed because it can change the meaning of mental health text
NEGATION_WORDS = {
    "no",
    "not",
    "never",
    "neither",
    "nor",
}

### Create the Preprocessing Function

In [58]:
"""Function removes spaces and punctuation replaces numbers with number converts tokens to lowercase lemmas removes stop words preserves important negation words keeps alphabetic tokens only"""


def preprocess_doc(doc) -> str:
    processed_tokens = []

    for token in doc:
        if token.is_space or token.is_punct:
            continue

        if token.like_num:
            processed_tokens.append("number")
            continue

        lemma = token.lemma_.lower().strip()

        if token.is_stop and lemma not in NEGATION_WORDS:
            continue

        if lemma.isalpha():
            processed_tokens.append(lemma)

    return " ".join(processed_tokens)

### Apply Tokenization and Lemmatization

In [59]:
# Create a column named TEXT_PROCESSED
docs = nlp.pipe(
    mental_health_df["TEXT_NORMALIZED"],
    batch_size=500,
)

mental_health_df["TEXT_PROCESSED"] = [preprocess_doc(doc) for doc in docs]

### Compare Normalized and Processed Text

In [60]:
mental_health_df[["TEXT_NORMALIZED", "TEXT_PROCESSED", "Label"]].sample(
    10, random_state=42
)

,TEXT_NORMALIZED,TEXT_PROCESSED,Label
12033,I Hope I Fucking Die Today.,hope fucking die today,2.0
3134,It's called the iAye.,call iaye,0.0
13051,I figured out that i cannot work. and money fr...,figure not work money family get dry gf strugg...,2.0
1662,Picking up women is like chess...,pick woman like chess,0.0
2481,Because they are more likely to be dead Wow. N...,likely dead wow understand rip inbox,0.0
12911,After my recent failed suicide attempt I can't...,recent fail suicide attempt not anymore body w...,2.0
903,I always buy groceries with cash.,buy grocery cash,0.0
13728,I have the razor ready to slice open my arms.,razor ready slice open arm,2.0
4282,What are you reading?,read,0.0
5285,Now I`m off to bed - HAPPY MOTHER`S DAY ALL - ...,bed happy day great number,0.0


### Validate the Processed Text

In [61]:
# Check for missing processed text
mental_health_df["TEXT_PROCESSED"].isna().sum()

np.int64(0)

In [62]:
# Count empty processed documents
(mental_health_df["TEXT_PROCESSED"].str.strip() == "").sum()

np.int64(87)

### Remove Empty Processed Documents

In [63]:
empty_processed = mental_health_df["TEXT_PROCESSED"].str.strip() == ""

print("Empty processed documents:", empty_processed.sum())

mental_health_df = mental_health_df.loc[~empty_processed].copy()

print("Final dataset shape:", mental_health_df.shape)

Empty processed documents: 87
Final dataset shape: (15774, 8)


### Preprocessing Summary

Text preprocessing produced a model-ready dataset by repairing encoding artifacts, normalizing text, removing duplicate and conflicting records, tokenizing the text, preserving important negation terms, removing stop words, and applying lemmatization. After preprocessing, 87 documents contained no remaining lexical content and were removed because they did not provide useful features for classification. The final preprocessed dataset contains **15,774 observations** and is ready for exploratory data analysis and feature engineering.

### Save the Preprocessed Dataset

In [64]:
output_file = PROCESSED_DATA_DIR / "mental_health_text_preprocessed.csv"

mental_health_df.to_csv(
    output_file,
    index=False,
)

print("Preprocessed dataset saved to:")
print(output_file)
print("Saved dataset shape:", mental_health_df.shape)

Preprocessed dataset saved to:
d:\Sowers, Sabriya\Education\NW\Capstone 6.29.26 - 8.14.26\capstone-nlp-mental-health\data\processed\mental_health_text_preprocessed.csv
Saved dataset shape: (15774, 8)
